**Quantum Phase Estimation (QPE).**
QPE is one of the most important subroutines in quantum computing.
Given a unitary operator $U$ and one of its eigenstates $|u\rangle$
satisfying $U|u\rangle = e^{2\pi i\phi}|u\rangle$,
QPE estimates the phase $\phi \in [0, 1)$ to $n-1$ bits of precision
using $n-1$ counting qubits and one target qubit.

The algorithm has three stages:

1.) **Initialization.**
Apply $H$ to all $n-1$ counting qubits to create equal superposition.
Prepare the target qubit in the eigenstate $|u\rangle = |1\rangle$.

2.) **Controlled-$U$ applications.**
In this notebook the unitary $U$ is a phase gate $P(\theta)$ satisfying
$P(\theta)\lvert 1\rangle = e^{i\theta}\lvert 1\rangle$,
so $\lvert 1\rangle$ is its eigenstate with eigenvalue $e^{i\theta}$.
Applying $U^{2^k}$ to the eigenstate is equivalent to a phase rotation
of $2^k\cdot\theta$, so each controlled-$U^{2^j}$ reduces to a single
$\text{CP}(2^{n-2-j}\cdot\theta)$ gate with the pre-multiplied angle.
Qubit $j=0$ receives the largest angle $2^{n-2}\cdot\theta$ (most significant
bit of the phase); each successive qubit receives half the angle of the
previous one.
After the inverse QFT transforms the counting register, the index of the
peak amplitude directly encodes $\phi = \theta/(2\pi)$ as an
$(n-1)$-bit binary fraction: $\hat{\phi} = \text{index}/2^{n-1}$.

3.) **Inverse QFT.**
The inverse Quantum Fourier Transform on the counting register converts
the phase information encoded in the amplitudes into a computational basis
state.
The index of the peak amplitude in the output statevector directly gives
the estimated phase:

$$\hat{\phi} = \frac{\text{index}}{2^{n-1}}$$

If $\phi$ is exactly representable as a binary fraction with $n-1$ bits,
the estimate is exact and the peak has probability 1.
Otherwise, the amplitude spreads across nearby indices and the estimate
has a small error bounded by $1/2^n$.

---
**Cell 01 - `qpe()` function, `iqft()` helper, and shared backend.**

**Why `iqft()` is implemented explicitly.**
Qiskit 2.1 deprecated the `QFT` class that previously provided the inverse QFT
via `QFT(n, do_swaps=False).inverse()`. The recommended replacement `QFTGate`
uses a different internal bit-ordering convention and does not expose a `do_swaps`
argument, which caused the peak index to land at the wrong position in the
statevector and produced completely incorrect phase estimates.
Rather than depend on any library class that may change convention between
Qiskit versions, the `iqft()` function constructs the inverse QFT directly
from primitive gates.

**What `iqft()` does.**
The inverse QFT reverses the standard QFT: it maps a state whose amplitudes
encode a phase into the computational basis state that represents that phase
as a binary fraction.
It is built from two primitives applied in reverse order to each qubit
from MSB to LSB:
1.) A Hadamard gate `H` converts the Fourier basis back to the computational basis.
2.) Controlled-phase gates `CP(-π/2^k)` undo the phase rotations that the
    forward QFT applied, where the negative sign reverses the direction of rotation.
No swap layer is included (`do_swaps=False` equivalent) because the bit-reversal
is handled implicitly by the QPE circuit structure.

**`qpe()` function.**
Takes the total number of qubits $n$ (counting qubits $= n-1$, target qubit $= 1$)
and the phase angle $\theta$ of the unitary $U$ where $U|1\rangle = e^{i\theta}|1\rangle$.
It builds the circuit, runs it, masks the target qubit out of the peak statevector
index, and reports the estimated phase $\hat{\phi} = \theta/(2\pi)$ along with
the percent relative error.
The circuit diagram is drawn only when $n \leq 6$ - larger circuits
are too wide to render usefully.

In [ ]:
"""qpe.ipynb"""

# Cell 01 - Quantum Phase Estimation function and shared backend

import numpy as np
from IPython.display import display
from qiskit import QuantumCircuit, transpile
from qiskit_aer import AerSimulator

backend = AerSimulator()


def iqft(num_qubits: int) -> QuantumCircuit:
    """Build an explicit inverse QFT circuit without final swaps.

    Constructs the inverse QFT as a sequence of Hadamard and controlled-phase
    gates without the bit-reversal swap layer, matching the convention required
    by the QPE counting register.

    Parameters
    ----------
    num_qubits : int
        Number of qubits for the inverse QFT.

    Returns
    -------
    QuantumCircuit
        The inverse QFT circuit.
    """
    qc = QuantumCircuit(num_qubits, name="IQFT")
    # Process from qubit 0 (MSB of phase in QPE) upward.
    # For each qubit k: first undo the phase rotations from
    # lower qubits, then apply H to convert to computational basis.
    for k in range(num_qubits):
        for j in range(k):
            qc.cp(-np.pi / 2 ** (k - j), j, k)
        qc.h(k)
    return qc


def qpe(n: int, theta: float) -> None:
    """Run Quantum Phase Estimation for a phase-gate unitary.

    Estimates the phase phi = theta / (2*pi) of the unitary
    U|1> = exp(i*theta)|1> using n-1 counting qubits and
    one target qubit.

    Parameters
    ----------
    n : int
        Total number of qubits. Counting qubits = n-1, target qubit = 1.
        Precision of estimate = n-1 bits, i.e. error < 1/2^n.
    theta : float
        Phase angle (radians) of the unitary: U|1> = exp(i*theta)|1>.
        The true phase is phi = theta / (2*pi).
    """
    qc = QuantumCircuit(n)

    # Hadamard on all counting qubits (0 to n-2)
    for qubit in range(n - 1):
        qc.h(qubit)

    # Target qubit (n-1) prepared in eigenstate |1>
    qc.x(n - 1)

    # Controlled-U^{2^j} gates: CP(2^(n-2-j) * theta) for j = 0..n-2
    # Each CP encodes one more binary digit of the phase
    for j in range(n - 1):
        angle = (2 ** (n - 2 - j)) * theta
        qc.cp(angle, j, n - 1)

    # Inverse QFT on counting register: converts phase -> computational basis
    qc.append(iqft(n - 1), range(n - 1))

    qc.save_statevector("sv")

    # Draw circuit only for small n (large circuits are unreadable)
    if n <= 6:
        display(qc.decompose(reps=2).draw(output="mpl"))

    result = backend.run(transpile(qc, backend)).result()

    # The statevector has 2^n elements for all n qubits.
    # The target qubit (n-1) is the MSB of the index; strip it
    # with modulo to isolate the (n-1)-bit counting register value.
    statevector = result.data(0)["sv"]
    probabilities = np.abs(statevector) ** 2
    index = np.argmax(probabilities) % (2 ** (n - 1))
    binary_phase = format(index, "0" + str(n - 1) + "b")
    phase_estimated = int(binary_phase, 2) / 2 ** (n - 1)
    phase_true = theta / (2 * np.pi)

    print(f"Counting qubits : {n - 1}")
    print(f"True phase      : {phase_true:.8f}")
    print(f"Peak index      : {index}  ({binary_phase} in binary)")
    print(f"Estimated phase : {phase_estimated:.8f}")
    if phase_true != 0:
        rel_error = abs(phase_estimated - phase_true) / phase_true * 100
        print(f"Relative error  : {rel_error:.4f}%")
    else:
        print("Relative error  : N/A (true phase is zero)")


print(f"Backend name : {backend.name}")
print(f"Aer version  : {backend.configuration().backend_version}")


**Cell 02 - Estimate phase of $U$ where $\theta = \pi/2$, using 3 qubits.**
The true phase is $\phi = \theta/(2\pi) = 1/4 = 0.25$.
With 2 counting qubits the representable phases are $0, 1/4, 1/2, 3/4$,
so $\phi = 1/4$ is exactly representable.
The peak index should be 1 (binary `01`) and the estimate is exact.

In [ ]:
# Cell 02 - theta = pi/2, n = 3 (2 counting qubits)
# True phase phi = 1/4, exactly representable -> error = 0

qpe(n=3, theta=np.pi / 2)

**Cell 03 - Estimate phase of $U$ where $\theta = \pi/4$, using 4 qubits.**
The true phase is $\phi = 1/8 = 0.125$.
With 3 counting qubits the representable phases are multiples of $1/8$,
so $\phi = 1/8$ is exactly representable.
The peak index should be 1 (binary `001`) and the estimate is exact.

In [ ]:
# Cell 03 - theta = pi/4, n = 4 (3 counting qubits)
# True phase phi = 1/8, exactly representable -> error = 0

qpe(n=4, theta=np.pi / 4)

**Cell 04 - Estimate phase of $U$ where $\theta = \pi/2.143$, using 16 qubits.**
The true phase is $\phi = 1/(2 \times 2.143) \approx 0.23332$,
which is not exactly representable as a binary fraction.
With 15 counting qubits the resolution is $1/2^{15} \approx 3\times 10^{-5}$,
so the estimate is accurate to about 5 decimal places.
The circuit is too large to draw ($n=16$) so only the numerical
result is displayed.

In [ ]:
# Cell 04 - theta = pi/2.143, n = 16 (15 counting qubits)
# True phase phi = 1/(2*2.143) ~ 0.23332, not exactly representable
# Resolution = 1/2^15 ~ 3e-5 -> estimate accurate to ~5 decimal places
# Circuit draw suppressed (n=16 is too large to render)

qpe(n=16, theta=np.pi / 2.143)